# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.


**Data licensing status:** The augmentation and negative-audio steps below select documented replacement sources and export attribution records. The legacy Piper `.pt` model and product dependencies still require a separate review. This notebook is not a blanket commercial-use clearance.

Data preparation downloads approximately 15 GB of archives; allow at least 40 GB of free disk space. It requires the system `zip` command for the multipart FSD50K archive. See [the data preparation guide](../documentation/licensed_training_data.md).

---

**WSL 本地训练版说明**：
- 数据集固定在 `/mnt/e/data/wake_word`（原始压缩包已提前用脚本下载到此处，无需重新下载 15 GB）。
- 仓库、生成的特征和训练输出保留在 WSL 的 Linux 文件系统中（ext4 的训练 I/O 远快于 `/mnt` 挂载的 Windows 盘）。
- 首次准备数据需要系统 `zip` 命令：`sudo apt install zip`。


In [1]:
# 路径和唤醒词名称集中设置；修改后请从本单元格开始重新运行。
from pathlib import Path
import os

repo_path = Path("/home/seeker32/projects/micro-wake-word").expanduser().resolve()
data_dir = Path("/mnt/e/data/wake_word").expanduser().resolve()
wake_word_name = "hey_kamana"  # 用于文件和目录名称；发音在合成单元格中设置
sample_dir = repo_path / "generated_samples" / wake_word_name
features_dir = repo_path / "licensed_features_v1"
positive_dir = features_dir / f"positive_{wake_word_name}"
train_dir = repo_path / "trained_models" / f"{wake_word_name}_licensed_inputs"
config_path = repo_path / f"training_parameters_{wake_word_name}.yaml"
archive_path = repo_path / f"{wake_word_name}_run.tar.gz"
data_root = data_dir / "prepared"

os.chdir(repo_path)
assert (repo_path / "microwakeword/audio/licensed_data.py").is_file(), f"找不到仓库：{repo_path}"


/home/seeker32/projects/micro-wake-word
microwakeword/audio/licensed_data.py


In [2]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import importlib.util
import platform
import subprocess
import sys
from pathlib import Path

# uv-created environments may not include pip. Bootstrap it for this kernel.
print(f"Installing into: {sys.executable}")
if importlib.util.find_spec("pip") is None:
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version"],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(repo_path)],
    check=True,
)

Installing into: /home/seeker32/projects/micro-wake-word/.venv/bin/python
Obtaining file:///home/seeker32/projects/micro-wake-word
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for microwakeword (pyproject.toml): started
  Building editable for microwakeword (pyproject.toml): finished with status 'done'
  Created wheel for microwakeword: filename=microwakeword-0.1.0-0.editable-py3-none-any.whl size=9964 sha256=f09d8ceddd7c89e6365ddfae6d54baae563bee2af3fad04c6dcdce651c911cf9
  Stored in directory: /tmp/pip-ephem-wheel-cache-8


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip


CompletedProcess(args=['/home/seeker32/projects/micro-wake-word/.venv/bin/python', '-m', 'pip', 'install', '-e', '/home/seeker32/projects/micro-wake-word'], returncode=0)

In [3]:
# Generates 1 sample of the target word for manual verification.
target_word = 'hˈeɪ kɑmˈɑnɑ'  # Hey Kamana (hey kah-MAH-nah), in Piper IPA

import importlib.util
import subprocess
import sys
from pathlib import Path

from IPython.display import Audio

# Keep installation and synthesis in the notebook kernel's environment.
print(f"Synthesizing with: {sys.executable}")
if importlib.util.find_spec("pip") is None:
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)
# Pin the matching CUDA 12.8 builds tested on this WSL RTX 5070 Ti setup.
# An unconstrained torch install selected 2.15/cu132 and crashed in CUDA discovery.
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "torch==2.9.1+cu128", "torchaudio==2.9.1+cu128",
     "--index-url", "https://download.pytorch.org/whl/cu128"],
    check=True,
)
# Avoid installing the generator's separate augmentation CLI dependency pins.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "piper-tts==1.3.0"],
    check=True,
)

piper_dir = repo_path / "piper-sample-generator"
if not piper_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/rhasspy/piper-sample-generator", str(piper_dir)],
        check=True,
    )

model_path = piper_dir / "models/en_US-libritts_r-medium.pt"
if not model_path.exists():
    subprocess.run(
        ["wget", "-O", str(model_path),
         "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"],
        check=True,
    )

# check=True stops this cell before playback if synthesis fails.
subprocess.run(
    [sys.executable, "-X", "faulthandler", "-m", "piper_sample_generator", target_word,
     "--phoneme-input", "--model", str(model_path),
     "--max-samples", "1", "--batch-size", "1",
     "--output-dir", str(sample_dir)],
    cwd=piper_dir,
    check=True,
)
sample_path = sample_dir / "0.wav"
if not sample_path.is_file():
    raise FileNotFoundError(f"Piper did not generate the expected sample: {sample_path}")
Audio(filename=str(sample_path), autoplay=True)


Synthesizing with: /home/seeker32/projects/micro-wake-word/.venv/bin/python
Looking in indexes: https://download.pytorch.org/whl/cu128



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
DEBUG:__main__:Loading /home/seeker32/projects/micro-wake-word/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/1 complete
INFO:__main__:Done


In [ ]:
# Generates a larger amount of wake word samples.
# Run the single-sample cell first to verify the pronunciation and environment.
# Experiment with noise scales, pronunciations, and confusing negative phrases.
import subprocess
import sys
from pathlib import Path

piper_dir = repo_path / "piper-sample-generator"
subprocess.run(
    [sys.executable, "-X", "faulthandler", "-m", "piper_sample_generator", target_word,
     "--phoneme-input",
     "--model", str(piper_dir / "models/en_US-libritts_r-medium.pt"),
     "--max-samples", "1000", "--batch-size", "100",
     "--output-dir", str(sample_dir)],
    cwd=piper_dir,
    check=True,
)


In [ ]:
# Prepare replacement augmentation and negative audio, with per-file provenance.
# WSL: raw archives already sit in the configured data_dir (downloaded beforehand);
# they are moved into downloads/ so prepare_licensed_data reuses them instead of
# re-downloading ~15 GB. Needs the system `zip` command for the FSD50K parts.

import shutil
from pathlib import Path

from microwakeword.audio.licensed_data import (
    FSD_FILES,
    SPEECH_FILES,
    prepare_licensed_data,
)

assert shutil.which("zip"), "缺少系统 zip 命令：请先运行 sudo apt install zip"

downloads = data_dir / "downloads"
downloads.mkdir(exist_ok=True)
for name in [*FSD_FILES, "rirs_noises.zip",
             *(f"{subset}.tar.gz" for subset, _ in SPEECH_FILES.values())]:
    flat = data_dir / name
    if flat.is_file() and not (downloads / name).exists():
        flat.rename(downloads / name)

data_root = prepare_licensed_data(str(data_dir))
print(f"Audio: {data_root}")
print(f"License manifest: {data_root / 'manifest.csv'}")
print(f"Attribution: {data_root / 'ATTRIBUTION.md'}")


In [ ]:
# 内核重启后先重新运行顶部的路径设置单元格。
data_root = data_dir / "prepared"
assert (data_root / "manifest.json").is_file(), f"找不到已准备的数据：{data_root}"


In [ ]:
# Sets up the augmentations.
# To improve your model, experiment with these settings and use more sources of
# background clips.

from pathlib import Path

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

if not (data_root / "manifest.json").is_file():
    raise FileNotFoundError(
        f"No prepared data at {data_root.resolve()}. "
        "Run the data preparation cell from the repository directory first."
    )

clips = Clips(input_directory=str(sample_dir),
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
# Separate background pools keep held-out audio out of training augmentation.
augmenters = {
    split: Augmentation(
        augmentation_duration_s=3.2,
        augmentation_probabilities={
            "SevenBandParametricEQ": 0.1,
            "TanhDistortion": 0.1,
            "PitchShift": 0.1,
            "BandStopFilter": 0.1,
            "AddColorNoise": 0.1,
            "AddBackgroundNoise": 0.75,
            "Gain": 1.0,
            "RIR": 0.5,
        },
        impulse_paths=[str(data_root / "rirs")],
        background_paths=[str(data_root / "noise" / split)],
        background_min_snr_db=-5,
        background_max_snr_db=10,
        min_jitter_s=0.195,
        max_jitter_s=0.205,
    )
    for split in ("training", "validation", "testing")
}
augmenter = augmenters["training"]


In [ ]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, str(repo_path / "augmented_clip.wav"))

Audio(str(repo_path / "augmented_clip.wav"), autoplay=True)

In [ ]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = str(positive_dir)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,    # Window length is 30 ms
                                     augmenter=augmenters[split],
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenters[split],
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

In [ ]:
# Build replacement negative features from the audited raw audio.
# Keep every source split intact. Ambient evaluation uses original clips >=6s,
# without synthetic repetition; it is a proxy for real-device room recordings.

import json
from pathlib import Path
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.licensed_data import digest

negative_root = features_dir / "negative"
for corpus, pattern in (("speech", "*.flac"), ("noise", "*.wav")):
    for split in ("training", "validation", "testing"):
        source_dir = data_root / corpus / split
        negative_clips = Clips(
            input_directory=str(source_dir),
            file_pattern=pattern,
            repeat_clip_min_duration_s=6,
        )
        generator = SpectrogramGeneration(
            clips=negative_clips,
            step_ms=10,
            split_spectrogram_duration_s=5,
        )
        out_dir = negative_root / corpus / split
        out_dir.mkdir(parents=True, exist_ok=True)
        RaggedMmap.from_generator(
            out_dir=str(out_dir / "negative_mmap"),
            sample_generator=generator.spectrogram_generator(),
            batch_size=100,
            verbose=True,
        )
        if split != "training":
            ambient_clips = Clips(
                input_directory=str(source_dir),
                file_pattern=pattern,
                min_clip_duration_s=6,
            )
            if len(ambient_clips.clips) == 0:
                raise ValueError(f"No original clips >=6s for ambient evaluation: {source_dir}")
            ambient_generator = SpectrogramGeneration(clips=ambient_clips, step_ms=10)
            out_dir = negative_root / "ambient" / f"{split}_ambient"
            out_dir.mkdir(parents=True, exist_ok=True)
            RaggedMmap.from_generator(
                out_dir=str(out_dir / f"{corpus}_mmap"),
                sample_generator=ambient_generator.spectrogram_generator(),
                batch_size=100,
                verbose=True,
            )

# Link derived features to the exact original-file license manifest.
lineage = {
    "audio_manifest": str((data_root / "manifest.json").resolve()),
    "audio_manifest_sha256": digest(data_root / "manifest.json"),
    "negative_features": str(negative_root),
    "positive_features": str(positive_dir),
    "processing": "16 kHz mono on access; 10 ms feature step; 5s negative chunks; "
                  "short classification negatives repeated to >=6s; ambient clips not repeated",
    "positive_augmentation": {split: {
        "background_pool": str(data_root / "noise" / split),
        "rir_pool": str(data_root / "rirs"),
    } for split in ("training", "validation", "testing")},
    "pending": "Legacy TTS provenance, dependency audit and real-device evaluation",
}
(negative_root.parent / "feature_sources.json").write_text(json.dumps(lineage, indent=2) + "\n")


In [ ]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = str(train_dir)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": str(positive_dir),
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": str(features_dir / "negative" / "speech"),
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": str(features_dir / "negative" / "noise"),
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # Only used for validation and testing
        "features_dir": str(features_dir / "negative" / "ambient"),
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]



# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
config["training_steps"] = [10000]
config["learning_rates"] = [0.001]  # Learning rates for Adam optimizer - list that corresponds to training steps
config["batch_size"] = 128

config["time_mask_max_size"] = [0]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (500)  # Test the validation sets after every this many stepsz
config["clip_duration_ms"] = (1500)  # Maximum length of wake word that the streaming model will accept


# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

with open(config_path, "w") as file:
    documents = yaml.dump(config, file)

In [ ]:
# Trains a model. When finished, it will quantize and convert the model to a
# streaming version suitable for on-device detection.
# It will resume if stopped, but it will start over at the configured training
# steps in the yaml file.
# Change --train 0 to only convert and test the best-weighted model.
# On Google colab, it doesn't print the mini-batch results, so it may appear
# stuck for several minutes! Additionally, it is very slow compared to training
# on a local GPU.

import sys

!"{sys.executable}" -m microwakeword.model_train_eval \
--training_config="{config_path}" \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

In [ ]:
# WSL 中没有 Google Colab 的 files.download；训练完成后模型在 Linux 文件系统内，
# 运行下面单元格得到 Windows 资源管理器可直接打开的 \\wsl$ 路径，手动复制即可。
import subprocess

model = str(train_dir / "tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")
print(subprocess.run(["wslpath", "-w", model], capture_output=True, text=True).stdout.strip())


In [ ]:
# 打包训练结果和数据来源记录。
import subprocess

subprocess.run([
    "tar", "-czf", str(archive_path),
    "-C", str(repo_path),
    str(train_dir.relative_to(repo_path)),
    str(config_path.relative_to(repo_path)),
    str((features_dir / "feature_sources.json").relative_to(repo_path)),
    "-C", str(data_dir),
    "prepared/manifest.json", "prepared/manifest.csv", "prepared/ATTRIBUTION.md",
    "prepared/excluded_fsd.csv", "prepared/sources",
], check=True)
print(f"已创建：{archive_path}")
